# Problema 2: Erdős-Rényi $G_{n,p}$ y los valores de $k_{min}$ y $k_{max}$

## 1. Desarrollo Teórico (Advanced Topics 3.B)

En una red aleatoria, la distribución de grados $p_k$ equivale en el límite de un $N$ grande a la distribución de Poisson:

$$ p_k = e^{-\langle k \rangle} \frac{\langle k \rangle^k}{k!} $$

El grado máximo $k_{max}$ se define como aquel en el cual esperamos que como máximo exista **solo un nodo** en toda la red con dicho grado. Matemáticamente esto se expresa condicionando su probabilidad a la inversa del tamaño de la red:

$$ N \cdot p_{k_{max}} \approx 1 \implies p_{k_{max}} \approx \frac{1}{N} $$

Sustituyendo la ecuación de Poisson:

$$ e^{-\langle k \rangle} \frac{\langle k \rangle^{k_{max}}}{k_{max}!} = \frac{1}{N} $$

Aplicando logaritmo natural a ambos lados de la ecuación:

$$ -\langle k \rangle + k_{max} \ln \langle k \rangle - \ln(k_{max}!) = -\ln N $$

Usando la aproximación de Stirling para el término del factorial ($\ln(x!) \approx x \ln x - x$), obtenemos:

$$ -\langle k \rangle + k_{max} \ln \langle k \rangle - k_{max} \ln(k_{max}) + k_{max} = -\ln N $$

$$ k_{max} \ln \left( \frac{\langle k \rangle e}{k_{max}} \right) = \langle k \rangle - \ln N $$

Con esta dependencia, a medida que $N$ aumenta, el crecimiento logarítmico dicta que tanto $k_{max}$ como $k_{min}$ (bajo la misma lógica $P(k_{min}) \approx 1/N$) se modificarán o expandirán apenas logarítmicamente $O(\ln N)$, manteniéndose sumamente restringidos pese al tamaño desproporcionado o inmenso de la red.


## 2. Generación, Cálculo Numérico y Tabulación

A continuación, mediante `scipy.stats.poisson.isf` (Inverse Survival Function) y `ppf` (Percent Point Function), calcularemos computacionalmente estos valores teóricos exactos equivalentes al umbral de un elemento en $N$ (`1/N`), para luego compararlos contra redes reales generadas velozmente con `nx.fast_gnp_random_graph`.


In [1]:
import networkx as nx
import numpy as np
import pandas as pd
from scipy.stats import poisson

N_values = [10000, 20000, 50000]
k_avg = 50

resultados = []

for N in N_values:
    p = k_avg / (N - 1)
    
    # Generar la red grande con memoria optimizada
    G = nx.fast_gnp_random_graph(N, p, seed=42)
    
    # Valores Empíricos
    grados = [d for _, d in G.degree()]
    k_min_emp = min(grados)
    k_max_emp = max(grados)
    
    # Valores Teóricos usando PPF (1/N inferior) y ISF (1/N superior)
    k_min_teo = poisson.ppf(1/N, mu=k_avg)
    k_max_teo = poisson.isf(1/N, mu=k_avg)
    
    resultados.append({
        'Tamaño (N)': N,
        'Prop. p': round(p, 5),
        'k_min (Teórico)': int(k_min_teo),
        'k_min (Empírico)': int(k_min_emp),
        'k_max (Teórico)': int(k_max_teo),
        'k_max (Empírico)': int(k_max_emp)
    })

# Tabular la comparativa
df_comparativa = pd.DataFrame(resultados)
display(df_comparativa)


,Tamaño (N),Prop. p,k_min (Teórico),k_min (Empírico),k_max (Teórico),k_max (Empírico)
0,10000,0.0050,26,24,78,78
1,20000,0.0025,25,26,80,81
2,50000,0.0010,24,24,82,85


## 3. Conclusión: ¿Qué se observa?

Al contrastar los valores empíricos de las redes generadas con las ecuaciones teóricas derivadas de la distribución de Poisson, podemos concluir tres puntos fundamentales sobre el modelo de Erdős-Rényi:

1. **Alta precisión del modelo teórico:** Los grados extremos empíricos ($k_{min}$ y $k_{max}$) se ajustan casi a la perfección a las cotas calculadas teóricamente asumiendo $P(k) \approx 1/N$. Las desviaciones mínimas (de 1 a 3 grados) corresponden a la varianza estocástica natural al instanciar redes aleatorias.
2. **Ausencia absoluta de Hubs (Estrechez de la distribución):** La distribución de Poisson decae exponencialmente rápido en sus colas. Esto queda en evidencia al observar que, incluso en una red masiva de 50,000 nodos donde cada uno tiene en promedio 50 conexiones, el nodo más conectado de toda la red tiene un grado máximo de apenas 85. En una red aleatoria, es estadísticamente imposible encontrar "hubs" desproporcionados (ej. nodos con miles de aristas).
3. **Crecimiento Logarítmico:** Queda demostrada la dependencia logarítmica de los grados extremos deducida en la sección teórica. Aunque el tamaño de la red se multiplicó por cinco (pasando de $N=10000$ a $N=50000$), el $k_{max}$ empírico apenas aumentó de 78 a 85. El tamaño de la red crece linealmente, pero los extremos de la distribución solo crecen a un ritmo $\mathcal{O}(\ln N)$.